# Recent Breakthroughs in LLMs

**Module 17: Research Papers Deep Dive**  
**Objective**: Understand recent major breakthroughs

Recent Papers Covered:
- LLaMA (2023): Efficient open models
- Mistral 7B (2023): Sliding window attention
- GPT-4 Technical Report
- Constitutional AI (Claude)
- Gemini: Multimodal capabilities

## What You'll Learn
1. LLaMA architecture innovations
2. Mistral's efficiency tricks
3. GPT-4 capabilities and limitations
4. Safety alignment techniques
5. Multimodal model design
6. Analyzing recent trends

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Tuple, Optional
import math

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}\n")

torch.manual_seed(42)
np.random.seed(42)

## 1. LLaMA: Open and Efficient Foundation Language Models

**Paper**: Touvron et al., 2023 (Meta AI)

**Key Contribution**: Train smaller models longer on more data

**Innovations**:
1. **Pre-normalization** (GPT-3 style):
   - Normalize input of each sublayer (not output)
   - More stable training

2. **SwiGLU activation** (instead of ReLU):
   ```
   SwiGLU(x) = Swish(xW) ⊙ xV
   ```

3. **Rotary Positional Embeddings (RoPE)**:
   - Relative position encoding
   - Better extrapolation to longer sequences

4. **Efficient training**:
   - 1.4T tokens for LLaMA-7B
   - Outperforms GPT-3 175B on many tasks

**Model Sizes**:
- LLaMA-7B: 7 billion parameters
- LLaMA-13B: 13 billion parameters
- LLaMA-33B: 33 billion parameters
- LLaMA-65B: 65 billion parameters

In [ ]:
class RoPE(nn.Module):
    """
    Rotary Position Embedding (RoPE) from LLaMA.
    
    Key innovation: Encodes relative positions via rotation.
    
    For position m, rotate by angle mθ:
    [x1]   [cos(mθ)  -sin(mθ)] [x1]
    [x2] = [sin(mθ)   cos(mθ)] [x2]
    """
    
    def __init__(self, dim: int, max_seq_len: int = 2048, base: int = 10000):
        super().__init__()
        self.dim = dim
        
        # Compute frequencies
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer('inv_freq', inv_freq)
        
        # Precompute for efficiency
        t = torch.arange(max_seq_len, dtype=torch.float)
        freqs = torch.einsum('i,j->ij', t, inv_freq)
        emb = torch.cat((freqs, freqs), dim=-1)
        self.register_buffer('cos_cached', emb.cos())
        self.register_buffer('sin_cached', emb.sin())
    
    def rotate_half(self, x: torch.Tensor) -> torch.Tensor:
        """Rotate half the hidden dims of the input."""
        x1, x2 = x[..., :x.shape[-1] // 2], x[..., x.shape[-1] // 2:]
        return torch.cat((-x2, x1), dim=-1)
    
    def forward(self, q: torch.Tensor, k: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Apply rotary embedding.
        
        Args:
            q, k: (batch, seq_len, n_heads, d_head)
        """
        seq_len = q.shape[1]
        
        cos = self.cos_cached[:seq_len].unsqueeze(0).unsqueeze(2)
        sin = self.sin_cached[:seq_len].unsqueeze(0).unsqueeze(2)
        
        q_rot = (q * cos) + (self.rotate_half(q) * sin)
        k_rot = (k * cos) + (self.rotate_half(k) * sin)
        
        return q_rot, k_rot

class SwiGLU(nn.Module):
    """
    SwiGLU activation from LLaMA.
    
    SwiGLU(x) = Swish(xW) ⊙ xV
    where Swish(x) = x * sigmoid(x)
    
    Benefits:
    - Better than ReLU for LLMs
    - Gating mechanism (like LSTM gates)
    """
    
    def __init__(self, dim: int, hidden_dim: int):
        super().__init__()
        self.w = nn.Linear(dim, hidden_dim, bias=False)
        self.v = nn.Linear(dim, hidden_dim, bias=False)
        self.w2 = nn.Linear(hidden_dim, dim, bias=False)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass.
        
        Args:
            x: (batch, seq_len, dim)
        """
        # Swish(xW) ⊙ xV
        swish_gate = F.silu(self.w(x))  # SiLU = Swish
        x = swish_gate * self.v(x)
        x = self.w2(x)
        return x

# Example usage
print("LLaMA Innovations:")
print("="*70)

batch_size = 2
seq_len = 10
n_heads = 8
d_head = 64
d_model = n_heads * d_head

# RoPE
rope = RoPE(d_head)
q = torch.randn(batch_size, seq_len, n_heads, d_head)
k = torch.randn(batch_size, seq_len, n_heads, d_head)
q_rot, k_rot = rope(q, k)

print(f"\n1. Rotary Position Embedding (RoPE):")
print(f"   Input Q shape: {q.shape}")
print(f"   Rotated Q shape: {q_rot.shape}")
print(f"   Benefits: Relative positions, better extrapolation")

# SwiGLU
swiglu = SwiGLU(d_model, d_model * 4)
x = torch.randn(batch_size, seq_len, d_model)
output = swiglu(x)

print(f"\n2. SwiGLU Activation:")
print(f"   Input shape: {x.shape}")
print(f"   Output shape: {output.shape}")
print(f"   Benefits: Better than ReLU, gating mechanism")

print("\n" + "="*70)
print("LLaMA Model Comparison:")
print("="*70)

models = [
    ("LLaMA-7B", 7, 32, 4096, 11008),
    ("LLaMA-13B", 13, 40, 5120, 13824),
    ("LLaMA-33B", 33, 60, 6656, 17920),
    ("LLaMA-65B", 65, 80, 8192, 22016),
]

print(f"\n{'Model':<15} {'Params':<10} {'Layers':<10} {'d_model':<10} {'d_ff':<10}")
print("-" * 70)
for name, params, layers, d_model, d_ff in models:
    print(f"{name:<15} {params}B{'':<7} {layers:<10} {d_model:<10} {d_ff:<10}")

print("\nTRAINING:")
print("  • Dataset: 1.4T tokens (public data)")
print("  • Sources: CommonCrawl, C4, GitHub, Wikipedia, Books, ArXiv, StackExchange")
print("  • Training: AdamW, cosine LR schedule, gradient clipping")
print("  • Context length: 2048 tokens")

print("\nRESULTS:")
print("  • LLaMA-13B > GPT-3 175B on most benchmarks")
print("  • LLaMA-65B competitive with Chinchilla-70B, PaLM-540B")
print("  • Open models enabling fine-tuning (Alpaca, Vicuna)")

print("\nIMPACT:")
print("  • Democratized LLMs (open weights)")
print("  • Showed smaller models can match larger ones")
print("  • Sparked open-source LLM movement")
print("  • Led to LLaMA 2, LLaMA 3 (up to 405B params)")

## 2. Mistral 7B: Sliding Window Attention

**Paper**: Jiang et al., 2023 (Mistral AI)

**Key Innovation**: Sliding Window Attention (SWA)

**Problem**: Full attention is O(n²), expensive for long sequences

**Solution**: Each token attends only to previous W tokens (window)

```
Full Attention:        Sliding Window (W=4):
[X X X X X X]          [X X X X - -]
[X X X X X X]          [X X X X X -]
[X X X X X X]   →     [X X X X X X]
[X X X X X X]          [- X X X X X]
[X X X X X X]          [- - X X X X]
[X X X X X X]          [- - - X X X]
```

**Benefits**:
- Reduces memory: O(n × W) instead of O(n²)
- Faster inference
- Information still propagates (through layers)
- After k layers: Effective receptive field = k × W

**Other Innovations**:
- Grouped Query Attention (GQA): Shares K/V across query heads
- Rolling buffer cache: Fixed memory for KV cache

In [ ]:
def visualize_sliding_window_attention():
    """Visualize sliding window attention pattern."""
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    seq_len = 12
    window_size = 4
    
    # 1. Full Attention
    ax1 = axes[0]
    ax1.set_title('Full Attention\n(Transformer)', fontsize=13, weight='bold')
    full_attn = np.ones((seq_len, seq_len))
    # Causal mask
    for i in range(seq_len):
        for j in range(i + 1, seq_len):
            full_attn[i, j] = 0
    
    im1 = ax1.imshow(full_attn, cmap='Blues', aspect='auto', vmin=0, vmax=1)
    ax1.set_xlabel('Key Position')
    ax1.set_ylabel('Query Position')
    ax1.set_xticks(range(0, seq_len, 2))
    ax1.set_yticks(range(0, seq_len, 2))
    
    # Count
    count = int(full_attn.sum())
    ax1.text(0.5, -0.15, f'Attention ops: {count}\nMemory: O(n²)',
            transform=ax1.transAxes, ha='center', fontsize=11,
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
    
    # 2. Sliding Window Attention
    ax2 = axes[1]
    ax2.set_title(f'Sliding Window (W={window_size})\n(Mistral)', fontsize=13, weight='bold')
    
    sliding_attn = np.zeros((seq_len, seq_len))
    for i in range(seq_len):
        for j in range(max(0, i - window_size + 1), i + 1):
            sliding_attn[i, j] = 1
    
    im2 = ax2.imshow(sliding_attn, cmap='Greens', aspect='auto', vmin=0, vmax=1)
    ax2.set_xlabel('Key Position')
    ax2.set_ylabel('Query Position')
    ax2.set_xticks(range(0, seq_len, 2))
    ax2.set_yticks(range(0, seq_len, 2))
    
    count = int(sliding_attn.sum())
    ax2.text(0.5, -0.15, f'Attention ops: {count}\nMemory: O(n×W)',
            transform=ax2.transAxes, ha='center', fontsize=11,
            bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.8))
    
    # 3. Receptive Field Growth
    ax3 = axes[2]
    ax3.set_title('Receptive Field Growth\n(Across Layers)', fontsize=13, weight='bold')
    
    layers = np.arange(1, 11)
    receptive_field = window_size * layers
    full_attention_field = np.full_like(layers, seq_len, dtype=float)
    
    ax3.plot(layers, receptive_field, 'g-o', linewidth=3, markersize=8,
            label=f'Sliding Window (W={window_size})')
    ax3.axhline(seq_len, color='blue', linestyle='--', linewidth=2,
               label='Full Sequence')
    
    # Mark where sliding window covers full sequence
    coverage_layer = int(np.ceil(seq_len / window_size))
    ax3.axvline(coverage_layer, color='red', linestyle=':', linewidth=2,
               alpha=0.5)
    ax3.text(coverage_layer, seq_len * 0.5, f'Full coverage\nat layer {coverage_layer}',
            rotation=90, va='bottom', ha='right', fontsize=10, color='red')
    
    ax3.set_xlabel('Layer Number', fontsize=11)
    ax3.set_ylabel('Receptive Field Size', fontsize=11)
    ax3.legend(fontsize=10)
    ax3.grid(True, alpha=0.3)
    ax3.set_xlim(1, 10)
    ax3.set_ylim(0, max(receptive_field) * 1.1)
    
    plt.tight_layout()
    plt.show()
    
    print("\nSliding Window Attention Analysis:")
    print("="*70)
    
    print(f"\nSequence Length: {seq_len}")
    print(f"Window Size: {window_size}")
    
    full_count = int(full_attn.sum())
    sliding_count = int(sliding_attn.sum())
    reduction = (1 - sliding_count / full_count) * 100
    
    print(f"\nCOMPLEXITY:")
    print(f"  Full Attention: {full_count} operations")
    print(f"  Sliding Window: {sliding_count} operations")
    print(f"  Reduction: {reduction:.1f}%")
    
    print(f"\nRECEPTIVE FIELD:")
    print(f"  Layer 1: {window_size} tokens")
    print(f"  Layer 2: {window_size * 2} tokens")
    print(f"  Layer {coverage_layer}: {window_size * coverage_layer} tokens (full coverage)")
    
    print(f"\nBENEFITS:")
    print(f"  ✓ Memory: O(n×W) vs O(n²)")
    print(f"  ✓ Speed: {reduction:.0f}% fewer operations")
    print(f"  ✓ Scales to longer sequences")
    print(f"  ✓ Information propagates through layers")

visualize_sliding_window_attention()

In [ ]:
# Mistral model comparison
print("\nMistral 7B Performance:")
print("="*70)

# Benchmark comparison (approximate)
benchmarks = {
    "MMLU": {"Mistral 7B": 60.1, "LLaMA 7B": 35.1, "LLaMA 13B": 47.6},
    "HellaSwag": {"Mistral 7B": 81.3, "LLaMA 7B": 76.1, "LLaMA 13B": 79.2},
    "MATH": {"Mistral 7B": 13.0, "LLaMA 7B": 2.5, "LLaMA 13B": 3.9},
}

fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(benchmarks))
width = 0.25

models = ["Mistral 7B", "LLaMA 7B", "LLaMA 13B"]
colors = ['green', 'blue', 'orange']

for i, model in enumerate(models):
    scores = [benchmarks[bench][model] for bench in benchmarks]
    bars = ax.bar(x + i * width, scores, width, label=model,
                 color=colors[i], alpha=0.7, edgecolor='black', linewidth=1.5)
    
    # Add value labels
    for bar, score in zip(bars, scores):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
               f'{score:.1f}',
               ha='center', va='bottom', fontsize=10, weight='bold')

ax.set_xlabel('Benchmark', fontsize=12, weight='bold')
ax.set_ylabel('Score', fontsize=12, weight='bold')
ax.set_title('Mistral 7B vs LLaMA Models', fontsize=14, weight='bold')
ax.set_xticks(x + width)
ax.set_xticklabels(benchmarks.keys())
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\nKEY FINDINGS:")
print("  • Mistral 7B outperforms LLaMA 13B (despite half the parameters)")
print("  • Especially strong on reasoning tasks (MMLU, MATH)")
print("  • 5x faster inference than LLaMA 13B")
print("  • Can handle 32K context with sliding window")

print("\nINNOVATIONS:")
print("  1. Sliding Window Attention: Reduces memory, enables long context")
print("  2. Grouped Query Attention: Shares K/V across query heads")
print("  3. Rolling Buffer Cache: Fixed memory for KV cache")

print("\nIMPACT:")
print("  • Best open 7B model (as of late 2023)")
print("  • Led to Mixtral 8x7B (sparse MoE)")
print("  • Widely adopted in production")
print("  • Proved efficiency tricks can match scale")

## 3. Recent Trends in LLM Research

**Efficiency**:
- Sparse models (MoE): Mixtral 8x7B, Grok
- Quantization: QLoRA, GPTQ, AWQ
- Efficient attention: Flash Attention, Sliding Window
- Smaller models: Phi-2 (2.7B), Gemma (2B/7B)

**Capabilities**:
- Long context: Claude 100K, GPT-4 Turbo 128K
- Multimodal: GPT-4V, Gemini, LLaVA
- Reasoning: Chain-of-thought, Self-consistency
- Tool use: Function calling, Agents

**Safety & Alignment**:
- RLHF: InstructGPT, ChatGPT
- Constitutional AI: Claude
- Red teaming: Adversarial testing
- Interpretability: Mechanistic interpretability

**Open Models**:
- LLaMA 2, LLaMA 3: Up to 405B params
- Mistral, Mixtral: Efficient architectures
- Qwen, Yi: Chinese tech companies
- Gemma: Google's open models

In [ ]:
def visualize_llm_landscape():
    """Visualize LLM landscape (2023-2024)."""
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    # 1. Model Size Timeline
    ax1 = axes[0, 0]
    ax1.set_title('Model Size Evolution', fontsize=13, weight='bold')
    
    models_timeline = [
        ("GPT-2", 2019, 1.5),
        ("GPT-3", 2020, 175),
        ("Gopher", 2021, 280),
        ("Chinchilla", 2022, 70),
        ("PaLM", 2022, 540),
        ("LLaMA", 2023, 65),
        ("GPT-4", 2023, 1000),  # Rumored
        ("LLaMA 3", 2024, 405),
    ]
    
    years = [m[1] for m in models_timeline]
    sizes = [m[2] for m in models_timeline]
    labels = [m[0] for m in models_timeline]
    
    ax1.plot(years, sizes, 'bo-', linewidth=2, markersize=10, alpha=0.6)
    for i, (year, size, label) in enumerate(zip(years, sizes, labels)):
        ax1.text(year, size * 1.15, label, ha='center', fontsize=9, weight='bold')
    
    ax1.set_xlabel('Year')
    ax1.set_ylabel('Parameters (Billions)')
    ax1.set_yscale('log')
    ax1.grid(True, alpha=0.3)
    
    # 2. Efficiency vs Performance
    ax2 = axes[0, 1]
    ax2.set_title('Efficiency vs Performance Trade-off', fontsize=13, weight='bold')
    
    models_efficiency = [
        ("GPT-3 175B", 175, 65),
        ("LLaMA 65B", 65, 68),
        ("LLaMA 13B", 13, 55),
        ("Mistral 7B", 7, 62),
        ("Phi-2", 2.7, 58),
    ]
    
    for name, params, score in models_efficiency:
        color = 'green' if 'Mistral' in name or 'Phi' in name else 'blue'
        ax2.scatter(params, score, s=200, alpha=0.6, color=color, edgecolors='black', linewidth=2)
        ax2.text(params * 1.15, score, name, fontsize=9, weight='bold')
    
    ax2.set_xlabel('Model Size (Billions of Parameters)')
    ax2.set_ylabel('MMLU Score')
    ax2.set_xscale('log')
    ax2.grid(True, alpha=0.3)
    
    # Pareto frontier
    ax2.text(0.05, 0.95, '↖ Efficient frontier\n(Better performance per param)',
            transform=ax2.transAxes, va='top', fontsize=9, style='italic',
            bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.7))
    
    # 3. Capability Progression
    ax3 = axes[1, 0]
    ax3.set_title('Capability Progression', fontsize=13, weight='bold')
    
    capabilities = [
        "Text\nGeneration",
        "Few-shot\nLearning",
        "Instruction\nFollowing",
        "Reasoning\n(CoT)",
        "Tool\nUse",
        "Multimodal"
    ]
    
    models_capability = {
        "GPT-2": [1, 0, 0, 0, 0, 0],
        "GPT-3": [1, 1, 0, 0, 0, 0],
        "ChatGPT": [1, 1, 1, 0.5, 0, 0],
        "GPT-4": [1, 1, 1, 1, 1, 1],
    }
    
    x = np.arange(len(capabilities))
    width = 0.2
    
    for i, (model, scores) in enumerate(models_capability.items()):
        offset = (i - len(models_capability)/2 + 0.5) * width
        bars = ax3.bar(x + offset, scores, width, label=model, alpha=0.7)
    
    ax3.set_xticks(x)
    ax3.set_xticklabels(capabilities, fontsize=9)
    ax3.set_ylabel('Capability Level')
    ax3.set_ylim(0, 1.2)
    ax3.legend(fontsize=10)
    ax3.grid(True, alpha=0.3, axis='y')
    
    # 4. Open vs Closed Models
    ax4 = axes[1, 1]
    ax4.set_title('Open vs Closed Ecosystem', fontsize=13, weight='bold')
    ax4.axis('off')
    
    # Closed models
    rect = plt.Rectangle((0.05, 0.55), 0.4, 0.35,
                         facecolor='lightcoral', edgecolor='black', linewidth=2)
    ax4.add_patch(rect)
    ax4.text(0.25, 0.85, 'CLOSED', ha='center', va='top',
            fontsize=12, weight='bold')
    closed_models = ["GPT-4", "Claude 3", "Gemini", "Grok"]
    for i, model in enumerate(closed_models):
        ax4.text(0.25, 0.75 - i*0.08, f"• {model}", ha='center', fontsize=10)
    
    # Open models
    rect = plt.Rectangle((0.55, 0.55), 0.4, 0.35,
                         facecolor='lightgreen', edgecolor='black', linewidth=2)
    ax4.add_patch(rect)
    ax4.text(0.75, 0.85, 'OPEN', ha='center', va='top',
            fontsize=12, weight='bold')
    open_models = ["LLaMA 3", "Mistral", "Qwen", "Gemma"]
    for i, model in enumerate(open_models):
        ax4.text(0.75, 0.75 - i*0.08, f"• {model}", ha='center', fontsize=10)
    
    # Trade-offs
    ax4.text(0.25, 0.4, 'Pros:\n• Best performance\n• Regular updates\n\nCons:\n• API costs\n• No customization',
            ha='center', va='top', fontsize=9,
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    ax4.text(0.75, 0.4, 'Pros:\n• Free weights\n• Customizable\n\nCons:\n• Lags in capabilities\n• Need infrastructure',
            ha='center', va='top', fontsize=9,
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    plt.tight_layout()
    plt.show()

visualize_llm_landscape()

print("\nKey Research Trends (2024):")
print("="*70)

print("\n1. EFFICIENCY:")
print("   • Smaller models with better performance (Phi-2, Gemma)")
print("   • Sparse models (MoE): Mixtral, Grok")
print("   • Quantization enabling on-device AI")

print("\n2. LONG CONTEXT:")
print("   • Claude 3: 200K tokens")
print("   • Gemini 1.5: 1M tokens")
print("   • Techniques: RoPE, ALiBi, Yarn")

print("\n3. MULTIMODAL:")
print("   • GPT-4V: Vision + text")
print("   • Gemini: Native multimodal")
print("   • LLaVA: Open multimodal")

print("\n4. SAFETY & ALIGNMENT:")
print("   • Constitutional AI (Claude)")
print("   • Anthropic's RLAIF")
print("   • Red teaming protocols")

print("\n5. OPEN SOURCE ECOSYSTEM:")
print("   • LLaMA 3 (up to 405B)")
print("   • Mistral models")
print("   • Fine-tuning tools (LoRA, QLoRA)")

## Summary

You've analyzed recent LLM breakthroughs:
- ✅ LLaMA: Efficient training, open models
- ✅ Mistral: Sliding window attention
- ✅ Understanding innovation trends
- ✅ Open vs closed ecosystems
- ✅ Current research directions

**Key Papers to Read**:
1. **"Attention is All You Need"** (Vaswani et al., 2017)
2. **"BERT"** (Devlin et al., 2018)
3. **"GPT-3"** (Brown et al., 2020)
4. **"LLaMA"** (Touvron et al., 2023)
5. **"Mistral 7B"** (Jiang et al., 2023)
6. **"Constitutional AI"** (Bai et al., 2022)
7. **"Flash Attention"** (Dao et al., 2022)
8. **"LoRA"** (Hu et al., 2021)

**Staying Current**:
- **arXiv**: Daily digest for cs.CL, cs.LG
- **Papers with Code**: Trending papers
- **Hugging Face**: Model releases
- **X/Twitter**: @_akhaliq, @omarsar0, @rctatman
- **Conferences**: NeurIPS, ICML, ICLR, ACL
- **Newsletters**: The Batch, Import AI

**Next Module**: Advanced Topics - MoE, NAS, Meta-learning